# Sesión 16: IA Ética y Sesgos en Modelos

**Objetivos:**
-   Entender qué es el **sesgo (bias)** en Machine Learning y de dónde proviene.
-   Conocer ejemplos reales del impacto social de los modelos sesgados.
-   Aprender a **auditar un modelo** para detectar si su rendimiento es desigual entre diferentes grupos demográficos.
-   Reflexionar sobre la responsabilidad ética del científico de datos.

## 1. ¿Qué Pasa Cuando los Algoritmos se Equivocan?

Un modelo puede tener una precisión del 95% y, aun así, ser profundamente injusto y perjudicial.

* **Caso COMPAS (Justicia):** Un algoritmo usado en EE. UU. para predecir la reincidencia criminal tenía el doble de probabilidad de etiquetar incorrectamente a acusados negros como "alto riesgo" que a acusados blancos.
* **Caso Amazon (Contratación):** Amazon desarrolló un modelo para filtrar currículums que aprendió de los datos históricos de la empresa y acabó penalizando sistemáticamente los CV que incluían la palabra "mujer".
* **Caso Reconocimiento Facial:** Muchos sistemas comerciales han demostrado tener tasas de error mucho más altas al identificar rostros de mujeres de piel oscura en comparación con hombres de piel clara.



Estos no son fallos técnicos aislados, son el resultado de un problema fundamental: el **sesgo**.

## 2. ¿De Dónde Viene el Sesgo? (Pista: No es el Algoritmo)

El sesgo en ML es un **error sistemático** que desfavorece a ciertos grupos. Generalmente, no nace del algoritmo en sí, sino de los datos con los que lo alimentamos.

* **Sesgo Histórico/Social:** Los datos reflejan sesgos y prejuicios del mundo real. Si históricamente se ha contratado a más hombres para puestos directivos, el modelo aprenderá que "ser hombre" es un indicador de ser un buen directivo.
* **Sesgo de Representación:** El dataset no representa por igual a todos los grupos. Si un modelo de reconocimiento facial se entrena mayoritariamente con fotos de personas blancas, funcionará mal para otras etnias.
* **Sesgo de Medición:** La forma en que medimos o etiquetamos los datos es imperfecta o injusta. Por ejemplo, usar el número de arrestos como un indicador de criminalidad puede reflejar más la intensidad de la vigilancia policial en un barrio que la actividad criminal real.

---
## Taller de Análisis Crítico: Auditoría de un Modelo de Ingresos

**Contexto:** Vamos a usar el dataset "Adult Census", cuyo objetivo es predecir si una persona gana más o menos de 50.000$ al año. Un banco podría usar un modelo similar para decidir a quién ofrecer productos financieros premium.

**Nuestra misión:** Entrenar un modelo, verificar su `accuracy` general y luego **auditarlo** para ver si es justo para todos los grupos.

### Actividad en Clase:
1.  **Carga y prepara** el dataset "Adult".
2.  **Entrena un `RandomForestClassifier`** y calcula su **`accuracy` general** en el conjunto de test.
3.  **Auditoría por Sexo:**
    * Filtra el conjunto de test para tener solo a los hombres y calcula la `accuracy` del modelo para ellos.
    * Haz lo mismo para las mujeres.
    * Compara ambas `accuracy`. ¿Es el modelo igual de preciso para ambos grupos?
4.  **Auditoría por Raza:**
    * Repite el proceso anterior, pero comparando la `accuracy` para personas de raza "Blanca" (`White`) con la de personas de raza "Negra" (`Black`).
    * ¿Qué observas?


In [ ]:
# --- ESPACIO PARA LA ACTIVIDAD EN CLASE ---
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# 1. Carga y preparación de datos
url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data'
columns = ['age', 'workclass', 'fnlwgt', 'education', 'education-num', 'marital-status', 
           'occupation', 'relationship', 'race', 'sex', 'capital-gain', 'capital-loss', 
           'hours-per-week', 'native-country', 'income']
df = pd.read_csv(url, header=None, names=columns, na_values=' ?', skipinitialspace=True)
df.dropna(inplace=True)
df_original = df.copy() # Guardamos una copia con los datos originales de texto

le = LabelEncoder()
df_encoded = df.apply(le.fit_transform)

X = df_encoded.drop('income', axis=1)
y = df_encoded['income']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [2]:
# 2. Entrenar modelo y obtener accuracy general
print("Entrenando Random Forest Classifier...")
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)

# Predicciones y accuracy general
y_pred = rf_model.predict(X_test)
accuracy_general = accuracy_score(y_test, y_pred)

print(f"\nAccuracy General del Modelo: {accuracy_general:.4f} ({accuracy_general*100:.2f}%)")

Entrenando Random Forest Classifier...

Accuracy General del Modelo: 0.8603 (86.03%)


In [5]:
# 3. Auditoría por Sexo 
# Usamos los índices del DataFrame original para identificar las filas de test que son hombres/mujeres

# Recuperar los datos originales de sexo para el conjunto de test
df_test_original = df_original.loc[X_test.index]

# Filtrar índices por sexo
idx_hombres = df_test_original[df_test_original['sex'] == 'Male'].index
idx_mujeres = df_test_original[df_test_original['sex'] == 'Female'].index

# Obtener predicciones y valores reales para cada grupo
y_test_hombres = y_test.loc[idx_hombres]
y_pred_hombres = rf_model.predict(X_test.loc[idx_hombres])

y_test_mujeres = y_test.loc[idx_mujeres]
y_pred_mujeres = rf_model.predict(X_test.loc[idx_mujeres])

# Calcular accuracy por sexo
accuracy_hombres = accuracy_score(y_test_hombres, y_pred_hombres)
accuracy_mujeres = accuracy_score(y_test_mujeres, y_pred_mujeres)

# Mostrar resultados
print("\n" + "="*60)
print("AUDITORÍA POR SEXO")
print("="*60)
print(f"Accuracy para Hombres: {accuracy_hombres:.4f} ({accuracy_hombres*100:.2f}%)")
print(f"Accuracy para Mujeres: {accuracy_mujeres:.4f} ({accuracy_mujeres*100:.2f}%)")
print(f"Diferencia: {abs(accuracy_hombres - accuracy_mujeres):.4f} ({abs(accuracy_hombres - accuracy_mujeres)*100:.2f}%)")

# Análisis
if accuracy_hombres > accuracy_mujeres:
    print(f"\nEl modelo es MÁS preciso para hombres (+{(accuracy_hombres - accuracy_mujeres)*100:.2f}%)")
else:
    print(f"\nEl modelo es MÁS preciso para mujeres (+{(accuracy_mujeres - accuracy_hombres)*100:.2f}%)")
    

# Información adicional
print(f"\nTamaño de muestras:")
print(f"  - hombres en test: {len(idx_hombres)}")
print(f"  - mujeres en test: {len(idx_mujeres)}")


AUDITORÍA POR SEXO
Accuracy para Hombres: 0.8243 (82.43%)
Accuracy para Mujeres: 0.9328 (93.28%)
Diferencia: 0.1085 (10.85%)

El modelo es MÁS preciso para mujeres (+10.85%)

Tamaño de muestras:
  - hombres en test: 4355
  - mujeres en test: 2158


In [4]:
# 4. Auditoría por Raza
# Seguimos el mismo patrón que la auditoría por sexo

# Filtrar índices por raza
idx_white = df_test_original[df_test_original['race'] == 'White'].index
idx_black = df_test_original[df_test_original['race'] == 'Black'].index

# Obtener predicciones y valores reales para cada grupo
y_test_white = y_test.loc[idx_white]
y_pred_white = rf_model.predict(X_test.loc[idx_white])

y_test_black = y_test.loc[idx_black]
y_pred_black = rf_model.predict(X_test.loc[idx_black])

# Calcular accuracy por raza
accuracy_white = accuracy_score(y_test_white, y_pred_white)
accuracy_black = accuracy_score(y_test_black, y_pred_black)

# Mostrar resultados
print("\n" + "="*60)
print("AUDITORÍA POR RAZA")
print("="*60)
print(f"Accuracy para Raza Blanca: {accuracy_white:.4f} ({accuracy_white*100:.2f}%)")
print(f"Accuracy para Raza Negra: {accuracy_black:.4f} ({accuracy_black*100:.2f}%)")
print(f"Diferencia: {abs(accuracy_white - accuracy_black):.4f} ({abs(accuracy_white - accuracy_black)*100:.2f}%)")

# Análisis
if accuracy_white > accuracy_black:
    print(f"\nEl modelo es MÁS preciso para personas blancas (+{(accuracy_white - accuracy_black)*100:.2f}%)")
else:
    print(f"\nEl modelo es MÁS preciso para personas negras (+{(accuracy_black - accuracy_white)*100:.2f}%)")

# Información adicional
print(f"\nTamaño de muestras:")
print(f"  - Personas blancas en test: {len(idx_white)}")
print(f"  - Personas negras en test: {len(idx_black)}")


AUDITORÍA POR RAZA
Accuracy para Raza Blanca: 0.8531 (85.31%)
Accuracy para Raza Negra: 0.9169 (91.69%)
Diferencia: 0.0639 (6.39%)

El modelo es MÁS preciso para personas negras (+6.39%)

Tamaño de muestras:
  - Personas blancas en test: 5533
  - Personas negras en test: 662


## Reflexión sobre los Resultados de Auditoría

### Hallazgos Principales

**Auditoría por Sexo:**
El modelo muestra una diferencia significativa del **10.85%** a favor de las mujeres. Esto significa que el modelo predice mejor si una mujer gana más o menos de 50.000$ que si lo hace un hombre. Esta disparidad es preocupante porque:

- Puede reflejar **sesgo de representación**: Hay casi el doble de hombres (4.355) que de mujeres (2.158) en el conjunto de test
- El dataset histórico probablemente contiene patrones más "predecibles" para mujeres (ej: menor variabilidad salarial debido a desigualdades estructurales del mercado laboral)
- Para los hombres, con mayor diversidad de situaciones laborales, el modelo comete más errores

**Auditoría por Raza:**
La diferencia del **6.39%** a favor de personas negras es paradójica. Sin embargo:

- La **desproporción en la muestra es crítica**: 5.533 personas blancas vs solo 662 personas negras (8:1)
- Este es un claro ejemplo de **sesgo de representación**: el modelo tiene muchos más datos para "aprender" sobre personas blancas
- La mayor precisión en personas negras puede deberse a patrones más homogéneos en sus datos debido a desigualdades históricas (menos variabilidad salarial, concentración en ciertos sectores)

### Conclusión Crítica

Aunque el accuracy general sea alto (~85%), estas diferencias demuestran que **el modelo no es justo**. Un banco que use este sistema discriminaría involuntariamente al cometer más errores con ciertos grupos, potencialmente negando oportunidades financieras de forma desigual.


### Reto para Casa: Reflexión
1.  Define con tus propias palabras **sesgo de representación** y **sesgo de medición**.

	El sesgo de respresentación es una desigualdad en los dato, haciendo que las predicciones salgan a favor d eun grupo u otro únicamente porque tenemos más datos de ese grupo. Para evitar este sesgo debemos tener datos equilibrados, con igual número de datos para cada grupo.

	El sesgo de medición ocurre cuando los datos que hemos recopilado no reflejan correctamente la realidad, por ejemplo si medimos la criminalidad en base al número de arrestos, una ciudad que esté muy vigilada tendría más arrestos, pero no por tener más criminalidad sino por tener más presencia policial.

2.  Nuestro modelo tenía una `accuracy` general alta, pero un rendimiento desigual. ¿Por qué esto demuestra que la `accuracy` por sí sola no es suficiente para evaluar un modelo en un contexto social?

	La accuracy general puede ocultar discriminaciones graves hacia grupos minoritarios. En nuestro caso tenemos más datos de hombres/blancos que de mujeres/negros, esto hace que el modelo no funcione igual de bien para todos. Generalizar en un contexto social puede ser peligroso porque nos lleva a perpetuar diferencias en la sociedad, lo ideal sería un modelo basado en equidad, evaluando por grupos separados y no de forma general

3.  Propón **dos acciones concretas** que podrías tomar sobre el *dataset* (pre-procesamiento) para intentar hacer el modelo más justo.

	**Eliminación de variables sensibles (Fairness-aware preprocessing)**
		- **Técnica:** Eliminar las columnas `sex` y `race` del dataset antes de entrenar el modelo
		- **Por qué:** Aunque parezca contraintuitivo, mantener estas variables puede hacer que el modelo aprenda patrones discriminatorios directamente
		- **Limitación importante:** Esto no es suficiente por sí solo, porque otras variables pueden actuar como "proxies" (ej: `occupation` puede estar correlacionado con género)
		- **Mejora adicional:** Combinar con técnicas de "debiasing" que detecten y eliminen correlaciones indirectas entre otras variables y los grupos protegidos

	**Pesos de clase personalizados**
		- Asignar mayor peso (`class_weight='balanced'` en RandomForest) a los errores cometidos en grupos minoritarios durante el entrenamiento, forzando al modelo a ser más cuidadoso con ellos

### Mis Reflexiones

*(Escribe aquí tus respuestas a las preguntas del reto para casa)*